In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

In [3]:
DRIVE = Path('/content/drive/MyDrive/WasteScope')
ARCHIVE_DIR = DRIVE / 'raw_archives'
TASKS = {
    'detection': ARCHIVE_DIR / 'detection_task.zip',
    'tracking': ARCHIVE_DIR / 'tracking_task.zip',
    'segmentation': ARCHIVE_DIR / 'segmentation_task.zip'
}
CLASSES = [
    'container_yellow', 'container_green', 'container_default',
    'container_blue', 'container_oil', 'container_battery',
    'container_biodegradable'
]

In [5]:
with ZipFile(TASKS["segmentation"]) as archive:
    json_name = next(name for name in archive.namelist() if Path(name).name == 'labels.json')
    coco = json.loads(archive.read(json_name).decode("utf-8-sig"))

categories = {item["id"]: item["name"] for item in coco["categories"]}
overflow_id = next(cid for cid, name in categories.items() if name.lower() == 'overflow')
first_container = min(cid for cid, name in categories.items() if name.startswith('container_'))

In [6]:
DETECTION_MAP = {
    cid - 1: CLASSES.index(name) for cid, name in categories.items() if name in CLASSES
}
TRACKING_MAP = {
    cid - first_container: CLASSES.index(name)
    for cid, name in categories.items() if name in CLASSES
}

DETECTION_SKIP = {cid - 1 for cid, name in categories.items() if name not in CLASSES}
TRACKING_SKIP = {
    cid - first_container for cid, name in categories.items()
    if name.startswith('container_') and name not in CLASSES
}